# Resume matching: a simple LangChain RAG pipeline

Following the supplied LangChain notebook: load documents, split text, create local Hugging Face embeddings, store vectors, retrieve context, and use a prompt -> OpenRouter LLM -> output parser chain.

We use **Chroma instead of FAISS** because Chroma is explicitly allowed by this assignment. Embeddings use `sentence-transformers/all-MiniLM-L6-v2` locally, as in the reference. Only optional job parsing and generated commentary use OpenRouter.

This notebook contains the dataset checks, chunking experiment, retrieval comparisons, measured accuracy and latency, example output, and conclusions. No separate evaluation framework is needed.

In [1]:
import json
import math
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

from resume_rag import ROOT, EMBEDDING_MODEL, load_resumes, chunk_resumes, build_index
from job_matcher import Requirements, match_jobs, meets_requirements

print('Measured at:', datetime.now(timezone.utc).isoformat())
print('Embeddings:', EMBEDDING_MODEL, '(local CPU)')

Measured at: 2026-10-21T07:11:38.922300+00:00
Embeddings: sentence-transformers/all-MiniLM-L6-v2 (local CPU)


## 1. Load the dataset with the Milestone 1 file readers

32 unique fictional candidates cover backend, frontend, data, ML, infrastructure, and QA roles at different seniority levels. There are 24 TXT, four PDF, and four DOCX resumes, plus six distinct job descriptions. Each job has a text version for OpenRouter parsing and a JSON version with explicit requirements for reproducible evaluation.

Metadata comes from explicit resume sections: name, skills, total years, skill-specific years, and education. Unknown years remain unknown (`-1` in flat Chroma metadata), and total years are never assigned to individual skills.

In [2]:
documents = load_resumes()
jobs = [json.loads(path.read_text(encoding='utf-8')) for path in sorted((ROOT / 'job_descriptions').glob('*.json'))]
labels = json.loads((ROOT / 'relevance_labels.json').read_text(encoding='utf-8'))['jobs']
paths = {doc.metadata['resume_path'] for doc in documents}
assert len(documents) >= 30 and len(paths) == len(documents)
assert len(jobs) >= 5
assert all(set(job['grades']) == paths for job in labels.values())
print('Unique resumes:', len(documents), '| Job descriptions:', len(jobs))
print('Formats:', {suffix: sum(Path(path).suffix == suffix for path in paths) for suffix in ['.txt', '.pdf', '.docx']})
print(json.dumps(documents[0].metadata, indent=2))

Unique resumes: 32 | Job descriptions: 6
Formats: {'.txt': 24, '.pdf': 4, '.docx': 4}
{
  "candidate_name": "Aarav Sharma",
  "resume_path": "c01_aarav_sharma.pdf",
  "skills": "[\"Python\", \"Django\", \"PostgreSQL\", \"Docker\", \"AWS\", \"REST APIs\"]",
  "experience_years": 8.0,
  "education": "[\"B.S. Computer Science, Example Institute 01\"]",
  "skill_years": "{\"python\": 7.0, \"django\": 6.0, \"postgresql\": 6.0, \"docker\": 5.0, \"aws\": 4.0, \"rest apis\": 7.0}"
}


## 2. Section-aware chunking experiment

Split on resume headings first, then use LangChain's `RecursiveCharacterTextSplitter` inside each section. This prevents Education and Experience from being mixed. Compare 500/50 and 700/100 character settings. The assignment dataset has short sections, so this experiment may show no material difference; we do not invent a benefit.

In [3]:
for size, overlap in [(500, 50), (700, 100)]:
    chunks = chunk_resumes(documents, size, overlap)
    lengths = [len(chunk.page_content) for chunk in chunks]
    assert max(lengths) <= size
    print({'chunk_size': size, 'overlap': overlap, 'chunks': len(chunks), 'mean_chars': round(statistics.mean(lengths), 1), 'max_chars': max(lengths)})
print('Default: 700/100, preserving the section as metadata.')

{'chunk_size': 500, 'overlap': 50, 'chunks': 224, 'mean_chars': 73.9, 'max_chars': 128}
{'chunk_size': 700, 'overlap': 100, 'chunks': 224, 'mean_chars': 73.9, 'max_chars': 128}
Default: 700/100, preserving the section as metadata.


## 3. Embed locally and persist in Chroma

The first run downloads the public embedding model. No OpenRouter credits are needed for indexing or local semantic retrieval. Rebuilding replaces this collection's chunks rather than appending duplicate resumes. This timing includes reading, chunking, model initialization/download, encoding, and persistence; it is separate from query latency.

In [4]:
started = time.perf_counter()
store = build_index()
indexing_seconds = time.perf_counter() - started
print('Stored chunks:', len(store.get()['ids']))
print('Indexing seconds:', round(indexing_seconds, 3))

C:\Users\320154882\Documents\LEARNING\AIRTRIBE\CODE\RAG-Based-Profile-matching\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2263.86it/s]

Stored chunks: 224
Indexing seconds: 34.228


## 4. Retrieval experimentation and performance metrics

Compare keyword skill coverage, semantic similarity, hybrid search, and hybrid search with strict must-haves. Hybrid score = **100 x (0.7 x semantic similarity + 0.3 x skill coverage)**. Semantic similarity is cosine mapped to [0,1]; coverage counts required and preferred skills mentioned in the candidate's skills section. Scores are relevance heuristics, not hiring probabilities. Aggregate chunks by resume before returning the top ten candidates.

Ground truth is author-curated in `relevance_labels.json` (0=not relevant/eligible, 1=partial, 2=good, 3=strong), fixed independently of retrieval output. Labels are never passed to the matcher.

- **Precision@10** divides relevant results by 10, even if filters return fewer candidates.
- **Recall@10** divides by all relevant candidates for that JD.
- **nDCG@10** measures the order of graded relevant results.
- **Returned precision** divides by actual returned results.
- **Must-have violation rate** measures returned candidates failing explicit requirements.
- Query **p50/p95 latency** includes local query embedding, Chroma search, candidate aggregation, filtering, scoring, and evidence formatting. It excludes index setup and OpenRouter generation. The first query is reported separately from repeated warm queries; query embeddings are not cached by this application.

In [5]:
def quality(ranking, grades):
    relevant = {path for path, grade in grades.items() if grade > 0}
    hits = sum(path in relevant for path in ranking[:10])
    dcg = sum((2 ** grades.get(path, 0) - 1) / math.log2(i + 2) for i, path in enumerate(ranking[:10]))
    ideal = sum((2 ** grade - 1) / math.log2(i + 2) for i, grade in enumerate(sorted(grades.values(), reverse=True)[:10]))
    return {'precision@10': hits / 10, 'recall@10': hits / len(relevant) if relevant else 0, 'ndcg@10': dcg / ideal if ideal else 0, 'returned_precision': hits / len(ranking) if ranking else 0}

assert quality(['a'], {'a': 3})['precision@10'] == 0.1
assert quality(['a'], {'a': 3})['recall@10'] == 1
variants = [('keyword', 'keyword', False), ('semantic', 'semantic', False), ('hybrid', 'hybrid', False), ('hybrid_filtered', 'hybrid', True)]
metadata_by_path = {doc.metadata['resume_path']: doc.metadata for doc in documents}
rows = []
for job in jobs:
    requirements = Requirements.model_validate(job['requirements'])
    for variant, mode, strict in variants:
        timings = []
        for repeat in range(4):
            started = time.perf_counter()
            result = match_jobs(store, job['description'], requirements, mode=mode, strict=strict, llm_reasoning=False)
            timings.append(1000 * (time.perf_counter() - started))
        ranking = [match['resume_path'] for match in result['top_matches']]
        assert len(ranking) == len(set(ranking)) <= 10
        violations = sum(not meets_requirements(metadata_by_path[path], requirements) for path in ranking)
        rows.append({'job': job['id'], 'variant': variant, **quality(ranking, labels[job['id']]['grades']), 'violation_rate': violations / len(ranking) if ranking else 0, 'first_ms': timings[0], 'warm_ms': timings[1:], 'ranking': ranking})
        if strict:
            assert violations == 0
print('Completed', len(rows), 'job/variant comparisons; strict must-have violations: 0')

Completed 24 job/variant comparisons; strict must-have violations: 0


In [6]:
import numpy as np

summary = {}
for variant, _, _ in variants:
    subset = [row for row in rows if row['variant'] == variant]
    timings = [value for row in subset for value in row['warm_ms']]
    summary[variant] = {metric: round(statistics.mean(row[metric] for row in subset), 4) for metric in ['precision@10', 'recall@10', 'ndcg@10', 'returned_precision', 'violation_rate']}
    summary[variant].update(p50_ms=round(statistics.median(timings), 2), p95_ms=round(float(np.percentile(timings, 95)), 2))
print(json.dumps(summary, indent=2))
print('\nPer-job results:')
for row in rows:
    print(row['job'], row['variant'], 'Recall@10:', round(row['recall@10'], 3), 'nDCG@10:', round(row['ndcg@10'], 3), 'first_ms:', round(row['first_ms'], 2))

{
  "keyword": {
    "precision@10": 0.2833,
    "recall@10": 1.0,
    "ndcg@10": 0.9504,
    "returned_precision": 0.2833,
    "violation_rate": 0.7167,
    "p50_ms": 16.88,
    "p95_ms": 18.85
  },
  "semantic": {
    "precision@10": 0.2833,
    "recall@10": 1.0,
    "ndcg@10": 0.9033,
    "returned_precision": 0.2833,
    "violation_rate": 0.7167,
    "p50_ms": 75.36,
    "p95_ms": 87.8
  },
  "hybrid": {
    "precision@10": 0.2833,
    "recall@10": 1.0,
    "ndcg@10": 0.9519,
    "returned_precision": 0.2833,
    "violation_rate": 0.7167,
    "p50_ms": 70.13,
    "p95_ms": 76.6
  },
  "hybrid_filtered": {
    "precision@10": 0.2833,
    "recall@10": 1.0,
    "ndcg@10": 0.9932,
    "returned_precision": 1.0,
    "violation_rate": 0.0,
    "p50_ms": 73.05,
    "p95_ms": 168.99
  }
}

Per-job results:
data_engineer keyword Recall@10: 1.0 nDCG@10: 0.959 first_ms: 14.98
data_engineer semantic Recall@10: 1.0 nDCG@10: 0.984 first_ms: 69.29
data_engineer hybrid Recall@10: 1.0 nDCG@10: 1.0 

## 5. Required JSON output and optional OpenRouter generation

The example below uses actual local semantic/hybrid retrieval and evidence-based explanations. Set `RUN_OPENROUTER=True` only when you want to call OpenRouter for generated explanations. The LangChain chain is `ChatPromptTemplate | ChatOpenAI | StrOutputParser` in `job_matcher.py`.

**OpenRouter parsing/generation is not benchmarked in this submission:** the earlier paid-model check returned HTTP 402 and the user requested no further live API measurements. This does not affect local embeddings or retrieval. No API response or API latency is fabricated.

In [7]:
RUN_OPENROUTER = False
example_job = next(job for job in jobs if job['id'] == 'python_backend')
example = match_jobs(store, example_job['description'], Requirements.model_validate(example_job['requirements']), llm_reasoning=RUN_OPENROUTER)
print(json.dumps(example, indent=2))

{
  "job_description": "Build transactional backend REST APIs using Python, Django and PostgreSQL. Must have Python, Django and PostgreSQL, at least 5 years total experience and 5+ years Python experience. Docker and AWS are preferred.",
  "top_matches": [
    {
      "candidate_name": "Aarav Sharma",
      "resume_path": "c01_aarav_sharma.pdf",
      "match_score": 93.03,
      "matched_skills": [
        "AWS",
        "Django",
        "Docker",
        "PostgreSQL",
        "Python"
      ],
      "relevant_excerpts": [
        "[Skills] Python, Django, PostgreSQL, Docker, AWS, REST APIs",
        "[Experience] Senior Backend Engineer, Fictional Studio 01\nBuilt order-processing APIs in Django with transactional PostgreSQL storage.",
        "[Skill Experience] Python: 7 years\nDjango: 6 years\nPostgreSQL: 6 years\nDocker: 5 years\nAWS: 4 years\nREST APIs: 7 years"
      ],
      "reasoning": "Matched sections: Skills, Experience, Skill Experience. Matched skills: AWS, Django, Dock

## 6. Analysis and limitations

The following conclusions are calculated from this execution, not assumed. On this small curated dataset, strict filters may dominate relevance because every positive label also satisfies the explicit job constraints. Inspect the unfiltered semantic and hybrid results before claiming retrieval quality. A low Precision@10 is expected when only two or three candidates are eligible: returning fewer than ten is correct, and returned precision makes the distinction visible.

Section-aware splitting preserves evidence but this small dataset cannot establish an optimal chunk size. Skill extraction expects an explicit Skills section; skill-specific years require explicit declarations. Scanned PDFs, date-based experience inference, and production-scale retrieval are out of scope. LLM commentary may still need human verification; deterministic scores and eligibility do not depend on it. Treat all candidates as fictional, and never send real personal resumes to OpenRouter without authorization.

In [8]:
filtered = summary['hybrid_filtered']
print('Filtered hybrid Recall@10:', filtered['recall@10'])
print('Filtered hybrid nDCG@10:', filtered['ndcg@10'])
print('Must-have violation rate:', filtered['violation_rate'])
print('Warm retrieval latency p50/p95:', filtered['p50_ms'], '/', filtered['p95_ms'], 'ms')
delta = summary['hybrid']['ndcg@10'] - summary['semantic']['ndcg@10']
print('Unfiltered hybrid minus semantic nDCG@10:', round(delta, 4))
print('OpenRouter generation: measured' if RUN_OPENROUTER else 'OpenRouter generation: NOT measured (explicitly disabled)')

Filtered hybrid Recall@10: 1.0
Filtered hybrid nDCG@10: 0.9932
Must-have violation rate: 0.0
Warm retrieval latency p50/p95: 73.05 / 168.99 ms
Unfiltered hybrid minus semantic nDCG@10: 0.0486
OpenRouter generation: NOT measured (explicitly disabled)
